# Rank Algebra — Five-Level Hierarchy

Demonstrates the `hllset-ranks` crate: FPGA-native integer rank propagation
from token frequency through bit, register, and HLLSet levels, plus dynamic
derivatives, Fisher matrix, and observable mask.

**Constraint:** all operations are integer (u64) — AND, OR, POPCOUNT, ADD, SUB, CMP.
No floats, no division.

In [2]:
:dep hllset-ranks = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-ranks" }
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }

In [3]:
use hllset_ranks::*;
use hllset_ranks::token::*;
use hllset_ranks::bit::*;
use hllset_ranks::register::*;
use hllset_ranks::hllset::*;
use hllset_ranks::compound::*;
use hllset_ranks::derivatives::*;
use hllset_ranks::fisher::*;
use hllset_ranks::mask::*;
use hllset_dsl::LatticeElement;
use hllset_core::HLLSet;
use std::collections::HashMap;

println!("hllset-ranks loaded — all values are u64 integers");
println!("FPGA-native operations: AND, OR, POPCOUNT, ADD, SUB, CMP");

hllset-ranks loaded — all values are u64 integers
FPGA-native operations: AND, OR, POPCOUNT, ADD, SUB, CMP


## Level 1: Token Rank — F(TF)

token-R = F(TF). Two implementations: Identity (F(x)=x) and Log2 (F(x)=floor(log2(x)) via LZCNT).

In [4]:
let identity = IdentityRankFn;
let log2 = Log2RankFn;

println!("Token Rank: F(TF) — Identity vs Log2");
println!("{:>8} {:>12} {:>12}", "TF", "identity", "log2");
println!("{}", "─".repeat(34));

for tf in [0, 1, 2, 5, 10, 50, 100, 1000, 10000, 100000] {
    let ri = identity.rank(tf);
    let rl = log2.rank(tf);
    println!("{:>8} {:>12} {:>12}", tf, ri, rl);
}

// Monotonicity check
let mut prev = 0u64;
let mut monotonic = true;
for tf in 0..1000 {
    let r = log2.rank(tf);
    if r < prev { monotonic = false; break; }
    prev = r;
}
println!("\nLog2 monotonic over 0..1000: {}", monotonic);

Token Rank: F(TF) — Identity vs Log2
      TF     identity         log2
──────────────────────────────────
       0            0            0
       1            1            0
       2            2            1
       5            5            2
      10           10            3
      50           50            5
     100          100            6
    1000         1000            9
   10000        10000           13
  100000       100000           16

Log2 monotonic over 0..1000: true


## Level 2: Bit Rank — G({token-R})

Multiple tokens may hash to the same (reg, tz) bit. G aggregates their token-ranks.
Max = dominant token controls the bit. Sum = all tokens contribute.

In [5]:
let max_g = MaxAggregator;
let sum_g = SumAggregator;

// Simulate 5 tokens colliding at (reg=42, tz=17) with different TFs
let tfs = [100u64, 5, 1000, 50, 2];
let tokens: Vec<TokenRank> = tfs.iter().map(|&tf| TokenRank::with_identity(tf)).collect();

let br_max = BitRank::new(42, 17, &tokens, &max_g);
let br_sum = BitRank::new(42, 17, &tokens, &sum_g);

println!("Bit (42, 17): {} tokens colliding", tokens.len());
println!("  Token TFs: {:?}", tfs);
println!("  Max bit-rank: {} (dominant token = {})", br_max.value, max_g.name());
println!("  Sum bit-rank: {} (all tokens = {})", br_sum.value, sum_g.name());

Bit (42, 17): 5 tokens colliding
  Token TFs: [100, 5, 1000, 50, 2]
  Max bit-rank: 1000 (dominant token = max)
  Sum bit-rank: 1157 (all tokens = sum)


## Level 3: Register Rank — H({bit-R})

A register spans 32 tz positions (tz=0..31). H aggregates their bit-ranks.
Three strategies: Sum, MaxPool, ActiveOnlySum (only bits set in HLLSet contribute).

In [6]:
let sum_h = SumRegAggregator;
let max_h = MaxPoolAggregator;
let active_h = ActiveOnlySum;

// Simulate 8 tz slots with bit-ranks in register 0
let bits: Vec<BitRank> = [
    (0, 10), (1, 200), (2, 50), (3, 5),
    (4, 300), (5, 15), (6, 0), (7, 100)
].iter().map(|&(tz, val)| BitRank { register: 0, tz, value: val, token_count: 1 }).collect();

let reg_sum = RegisterRank::new(0, &bits, &sum_h);
let reg_max = RegisterRank::new(0, &bits, &max_h);

// ActiveOnlySum with a mask where only tz=1,4,7 are set (bits 1,4,7 → mask = 2|16|128 = 146)
let mask: u32 = (1 << 1) | (1 << 4) | (1 << 7);
let reg_active = active_h.aggregate_masked(0, &bits, mask);

println!("Register 0: {} active tz slots", bits.len());
println!("  Sum:       {}", reg_sum.value);
println!("  MaxPool:   {}", reg_max.value);
println!("  ActiveOnly (mask 0b{:08b}): {}", mask, reg_active);
println!("  (Active bits tz=1(200) + tz=4(300) + tz=7(100) = {})", 200+300+100);

Register 0: 8 active tz slots
  Sum:       680
  MaxPool:   300
  ActiveOnly (mask 0b10010010): 600
  (Active bits tz=1(200) + tz=4(300) + tz=7(100) = 600)


## Level 4: HLLSet Rank — K(degree)

Structural importance in the lattice DAG. Default: degree (count of incident edges).
Alternative: weighted degree (sum of R-link popcounts).

In [7]:
let deg_k = DegreeRankFn;
let wdeg_k = WeightedDegreeRankFn;

// Create HLLSets and build a rank index
let alpha = LatticeElement::from_tokens(&["alpha", "beta", "gamma"]);
let beta  = LatticeElement::from_tokens(&["beta", "gamma", "delta"]);
let gamma = LatticeElement::from_tokens(&["gamma", "delta", "epsilon"]);

let mut idx = HLLSetRankIndex::new();

// Simulate lattice structure: alpha connected to beta and gamma (degree=2)
// beta connected to alpha and gamma (degree=2)
// gamma connected to alpha and beta (degree=2)
for (elem, deg) in [(&alpha, 2usize), (&beta, 2), (&gamma, 2)] {
    let r = HLLSetRank::new(elem, deg, &deg_k);
    idx.insert(r);
}

println!("HLLSet Rank Index: {} entries", idx.len());
for rank in idx.iter() {
    println!("  {}: degree={}, rank={}, popcount={}",
        &rank.key[..16], rank.degree, rank.value, rank.popcount);
}

// Weighted degree for alpha
let r_alpha_w = HLLSetRank::new(&alpha, 2, &wdeg_k);
println!("\nalpha weighted-degree rank: {} (popcount-based)", r_alpha_w.value);

HLLSet Rank Index: 3 entries
  h:e17e9c596e7521: degree=2, rank=2, popcount=3
  h:b7ed2db27e80dc: degree=2, rank=2, popcount=3
  h:67b9303fb55d2b: degree=2, rank=2, popcount=3

alpha weighted-degree rank: 3 (popcount-based)


## Level 5: Compound Rank — L(max) / M(min)

When HLLSets combine via lattice operations, ranks propagate.

In [8]:
let r_a = 100u64;
let r_b = 42u64;

let union = CompoundRank::union(r_a, r_b);
let inter = CompoundRank::intersection(r_a, r_b);
let diff  = CompoundRank::difference(r_a, r_b);
let sym   = CompoundRank::symmetric_difference(r_a, r_b);

println!("Operand ranks: A={}, B={}", r_a, r_b);
println!("  A ∪ B  = max({}, {}) = {}", r_a, r_b, union.value);
println!("  A ∩ B  = min({}, {}) = {}", r_a, r_b, inter.value);
println!("  A - B   = {} (inherits left)", diff.value);
println!("  A ⊕ B   = max({}, {}) = {}", r_a, r_b, sym.value);

Operand ranks: A=100, B=42
  A ∪ B  = max(100, 42) = 100
  A ∩ B  = min(100, 42) = 42
  A - B   = 100 (inherits left)
  A ⊕ B   = max(100, 42) = 100


## Dynamic Analysis: Noether Steering

The Noether controller monitors |card(N) - card(D)| → 0. With rank-weighted
steering, we also check |ΣR(N) - ΣR(D)| → 0.

In [9]:
// Two successive states: H(t-1) and H(t)
let prev = LatticeElement::from_tokens(&["cat", "dog", "bird", "fish"]);
let curr = LatticeElement::from_tokens(&["cat", "dog", "bird", "hamster"]);
// "fish" departed, "hamster" arrived — cat/dog/bird retained

// Bit-count steering (original Noether)
let ns_pop = NoetherSteering::compute_popcount(&prev, &curr, 5);
println!("Bit-count Noether:");
println!("  |N| - |D| divergence: {}", ns_pop.bit_divergence);
println!("  Structural equilibrium: {}", ns_pop.structural_equilibrium);

// Rank-weighted steering (each bit has rank = 10, except departed=5, new=15)
// In practice, rank_of_bit comes from the five-level hierarchy
let ns_rank = NoetherSteering::compute(
    &prev, &curr,
    &|reg: u32, _tz: u32| if reg < 512 { 10 } else { 5 },  // simplified
    5, 50
);
println!("\nRank-weighted Noether:");
println!("  Rank divergence: {}", ns_rank.rank_divergence);
println!("  Rank equilibrium: {}", ns_rank.rank_equilibrium);

Bit-count Noether:
  |N| - |D| divergence: 0
  Structural equilibrium: true

Rank-weighted Noether:
  Rank divergence: 0
  Rank equilibrium: true


## Dynamic Analysis: Rank Flux and Derivatives

ΔR(t) = velocity. Δ²R(t) = acceleration. The Noether controller reads acceleration
to decide whether to widen the window, adjust gates, or trigger reproduction.

In [10]:
// Simulate a flux history: system starts unstable, then settles
let flux_history = vec![50i64, 30, 15, 8, 3, 0, -2, 0, 1, 0];
let derivs = RankDerivatives::from_flux_history(&flux_history);

println!("Rank Flux and Derivatives");
println!("{:>4} {:>10} {:>12} {:>14}", "t", "flux", "velocity", "acceleration");
println!("{}", "─".repeat(44));
for d in &derivs {
    let acc = d.acceleration.map_or("  —".to_string(), |a| format!("{:>+4}", a));
    println!("{:>4} {:>+10} {:>+12} {:>14}", d.t, flux_history[d.t], d.velocity, acc);
}

// Check: did acceleration converge to near-zero?
let last_acc = derivs.last().and_then(|d| d.acceleration).unwrap_or(0);
println!("\nFinal acceleration: {} → system {}", last_acc,
    if last_acc.abs() <= 3 { "STABLE" } else { "DIVERGING" });

Rank Flux and Derivatives
   t       flux     velocity   acceleration
────────────────────────────────────────────
   0        +50           +0              —
   1        +30          -20              —
   2        +15          -15             +5
   3         +8           -7             +8
   4         +3           -5             +2
   5         +0           -3             +2
   6         -2           -2             +1
   7         +0           +2             +4
   8         +1           +1             -1
   9         +0           -1             -2

Final acceleration: -2 → system STABLE


## Fisher-Like Cross-Layer Matrix

F_{bb'} counts co-occurrence of bits across temporal layers. Bits that
appear together across many layers are functionally coupled — systemic
changes, not noise.

In [11]:
let mut fisher = FisherMatrix::new();

// Simulate 5 temporal layers, each with slightly different token sets
let layer_tokens = vec![
    vec!["alpha", "beta", "gamma"],           // L0: second
    vec!["alpha", "beta", "gamma", "delta"],  // L1: minute (delta joined)
    vec!["alpha", "beta", "delta"],            // L2: hour (gamma left)
    vec!["alpha", "beta", "gamma", "epsilon"], // L3: day (gamma back, epsilon new)
    vec!["alpha", "beta", "gamma"],            // L4: week (epsilon left)
];

for tokens in &layer_tokens {
    let layer = LatticeElement::from_tokens(tokens);
    fisher.add_layer(&layer);
}

println!("Fisher matrix: {} layers, {} non-zero entries",
    fisher.layer_count(), fisher.entry_count());

// Most persistent bits (appear in most layers)
let persistent = fisher.most_persistent(5);
println!("\nMost persistent bits (highest diagonal):");
for (pos, count) in &persistent {
    println!("  (reg={}, tz={}): appears in {}/{} layers", pos.0, pos.1, count, fisher.layer_count());
}

// Check coupling for the most persistent bit
if let Some((top_pos, _)) = persistent.first() {
    let coupled = fisher.most_coupled(*top_pos, 3);
    println!("\nBits most coupled to ({}, {}):", top_pos.0, top_pos.1);
    for (pos, count) in &coupled {
        println!("  ({}, {}): co-occurs in {} layers", pos.0, pos.1, count);
    }
}

Fisher matrix: 5 layers, 14 non-zero entries

Most persistent bits (highest diagonal):
  (reg=677, tz=0): appears in 5/5 layers
  (reg=661, tz=0): appears in 5/5 layers
  (reg=261, tz=0): appears in 4/5 layers
  (reg=120, tz=0): appears in 2/5 layers
  (reg=117, tz=0): appears in 1/5 layers

Bits most coupled to (677, 0):
  (661, 0): co-occurs in 5 layers
  (261, 0): co-occurs in 4 layers
  (120, 0): co-occurs in 2 layers


()

## Fisher Projection: Systemic vs Noise

s = F · d: project the divergence vector through the Fisher matrix.
High |s_b| means bit b's change is coupled to many other changes — systemic.
Low |s_b| means isolated fluctuation — noise.

In [12]:
// Simulate a divergence: some bits entered (+1), some left (-1), most stable (0)
let mut divergence: HashMap<BitPos, i8> = HashMap::new();

// Get positions from the last layer
let last_tokens = layer_tokens.last().unwrap().clone();
let last_layer = LatticeElement::from_tokens(&last_tokens);
let positions = last_layer.hllset().active_positions();

// Mark first 2 positions as "new" (+1), next 2 as "stable" (0),
// and simulate a departed bit (-1) at an arbitrary position
for (i, pos) in positions.iter().enumerate() {
    if i < 2 {
        divergence.insert(*pos, 1);
    } else if i == 2 {
        divergence.insert(*pos, 0);
    } else if i == 3 {
        divergence.insert(*pos, -1);
    }
}

let proj = FisherProjection::project(&fisher, &divergence, 2);

println!("Fisher Projection: {} bits scored", proj.scores.len());
println!("  Significant (|s| > 2): {} bits", proj.significant.len());
println!("\nTop scores:");
for (pos, score) in proj.scores.iter().take(8) {
    let tag = if score.abs() > 2 { " ← SYSTEMIC" } else { "" };
    println!("  (reg={:>4}, tz={:>2}): {:>+6}{}", pos.0, pos.1, score, tag);
}

Fisher Projection: 5 bits scored
  Significant (|s| > 2): 4 bits

Top scores:
  (reg= 661, tz= 0):    +14 ← SYSTEMIC
  (reg= 261, tz= 0):    +12 ← SYSTEMIC
  (reg= 677, tz= 0):     +9 ← SYSTEMIC
  (reg= 120, tz= 0):     +3 ← SYSTEMIC
  (reg= 117, tz= 0):     +2


()

## Observable Mask: O(θ) and Sub-Lattice Degree

The mask selects HLLSets above threshold. When ranks shift, the mask changes,
and sub-lattice degrees update — closing the feedback loop.

In [13]:
// Build a rank index with 5 HLLSets at varying ranks
let mut idx = HLLSetRankIndex::new();
let elements = vec![
    ("h:a", 100u64, 3usize),
    ("h:b", 80u64, 4usize),
    ("h:c", 60u64, 2usize),
    ("h:d", 40u64, 1usize),
    ("h:e", 20u64, 1usize),
];
for &(key, value, degree) in &elements {
    idx.insert(HLLSetRank::from_raw(key, degree, value * 10, &DegreeRankFn));
}
// Override values for the test
for rank in idx.iter() {
    // We'll just use the index as-is; values were set by from_raw with degree
}

// Actually, let's rebuild with explicit values
let mut idx2 = HLLSetRankIndex::new();
for &(key, value, degree) in &elements {
    let mut r = HLLSetRank::from_raw(key, degree, value * 10, &DegreeRankFn);
    r.value = value;
    idx2.insert(r);
}

let mask50 = ObservableMask::apply(&idx2, 50);
let mask30 = ObservableMask::apply(&idx2, 30);

println!("Observable Mask at θ=50:");
println!("  Observable: {} / {} HLLSets", mask50.observable_count(), mask50.total);
println!("  Hidden: {}", mask50.hidden.len());

println!("\nObservable Mask at θ=30:");
println!("  Observable: {} / {} HLLSets", mask30.observable_count(), mask30.total);

let diff = ObservableMask::diff(&mask50, &mask30);
println!("\nMask change (θ=50 → θ=30):");
println!("  Entered: {:?}", diff.entered);
println!("  Exited:  {:?}", diff.exited);
println!("  Churn:   {} HLLSets", diff.churn());

// Sub-lattice degree
let mut sub_deg = SubLatticeDegree::new();
sub_deg.add_edge("h:a", "h:b", &mask50);
sub_deg.add_edge("h:a", "h:c", &mask50);
sub_deg.add_edge("h:b", "h:d", &mask50);

println!("\nSub-lattice degrees at θ=50:");
println!("  h:a = {} (connected to b,c)", sub_deg.degree("h:a"));
println!("  h:b = {} (connected to a,d)", sub_deg.degree("h:b"));
println!("  h:d = {} (connected to b — but d is hidden at θ=50)", sub_deg.degree("h:d"));

Observable Mask at θ=50:
  Observable: 3 / 5 HLLSets
  Hidden: 2

Observable Mask at θ=30:
  Observable: 4 / 5 HLLSets

Mask change (θ=50 → θ=30):
  Entered: ["h:d"]
  Exited:  []
  Churn:   1 HLLSets

Sub-lattice degrees at θ=50:
  h:a = 2 (connected to b,c)
  h:b = 1 (connected to a,d)
  h:d = 0 (connected to b — but d is hidden at θ=50)


## Full Pipeline: Token → Bit → Register → HLLSet → Compound

End-to-end: tokenize text, compute ranks through all five levels, derive
compound rank for a union operation.

In [14]:
// Build two HLLSets from different token sets
let doc_a = LatticeElement::from_tokens(&["machine", "learning", "neural", "network"]);
let doc_b = LatticeElement::from_tokens(&["deep", "learning", "gradient", "network"]);

// Level 1: token ranks (simulated — in production, TF comes from LUT)
let tf_map: HashMap<&str, u64> = [
    ("machine", 10), ("learning", 50), ("neural", 30), ("network", 40),
    ("deep", 20), ("gradient", 15),
].iter().cloned().collect();

let f = IdentityRankFn;
let token_ranks: HashMap<&str, TokenRank> = tf_map.iter()
    .map(|(&t, &tf)| (t, TokenRank::new(tf, &f)))
    .collect();

println!("Token ranks (Level 1):");
for (token, tr) in &token_ranks {
    println!("  {:>10}: TF={:>3}, rank={:>3}", token, tr.tf, tr.value);
}

// Level 2-3: for each HLLSet, compute register ranks from active positions
// (In production, bit-ranks come from token→hash mapping; here we approximate)
let positions_a = doc_a.hllset().active_positions();
let positions_b = doc_b.hllset().active_positions();

// Simplified: each active position gets rank = popcount of the HLLSet in that register
let h_sum = SumRegAggregator;
let mut total_reg_rank_a: Rank = 0;
let mut total_reg_rank_b: Rank = 0;

// Group positions by register and sum their popcount-based ranks
let mut reg_ranks_a: HashMap<u32, Vec<BitRank>> = HashMap::new();
for &(reg, tz) in &positions_a {
    reg_ranks_a.entry(reg).or_default().push(BitRank { register: reg, tz, value: 1, token_count: 1 });
}
for (reg, bits) in &reg_ranks_a {
    total_reg_rank_a += h_sum.aggregate(*reg, bits);
}

let mut reg_ranks_b: HashMap<u32, Vec<BitRank>> = HashMap::new();
for &(reg, tz) in &positions_b {
    reg_ranks_b.entry(reg).or_default().push(BitRank { register: reg, tz, value: 1, token_count: 1 });
}
for (reg, bits) in &reg_ranks_b {
    total_reg_rank_b += h_sum.aggregate(*reg, bits);
}

println!("\nRegister ranks (Level 3, sum of active bits):");
println!("  doc_a: {} active positions → reg-rank sum = {}", positions_a.len(), total_reg_rank_a);
println!("  doc_b: {} active positions → reg-rank sum = {}", positions_b.len(), total_reg_rank_b);

// Level 4: HLLSet rank (degree-based — here we use popcount as proxy for degree)
// In production, degree comes from the lattice DAG
let rank_a = HLLSetRank::new(&doc_a, positions_a.len(), &DegreeRankFn);
let rank_b = HLLSetRank::new(&doc_b, positions_b.len(), &DegreeRankFn);

println!("\nHLLSet ranks (Level 4):");
println!("  doc_a: degree={}, rank={}", rank_a.degree, rank_a.value);
println!("  doc_b: degree={}, rank={}", rank_b.degree, rank_b.value);

// Level 5: compound — union and intersection
let union_elem = doc_a.union(&doc_b);
let inter_elem = doc_a.intersection(&doc_b);

let union_rank = CompoundRank::union(rank_a.value, rank_b.value);
let inter_rank = CompoundRank::intersection(rank_a.value, rank_b.value);

println!("\nCompound ranks (Level 5):");
println!("  A ∪ B: max({}, {}) = {}", rank_a.value, rank_b.value, union_rank.value);
println!("  A ∩ B: min({}, {}) = {}", rank_a.value, rank_b.value, inter_rank.value);
println!("  Union popcount: {}", union_elem.popcount());
println!("  Intersection popcount: {}", inter_elem.popcount());

// Noether check between A and B
let ns = NoetherSteering::compute_popcount(&doc_a, &doc_b, 50);
println!("\nNoether steering (A → B):");
println!("  |N-D| divergence: {}", ns.bit_divergence);
println!("  Structural equilibrium: {}", ns.structural_equilibrium);

Token ranks (Level 1):
     network: TF= 40, rank= 40
        deep: TF= 20, rank= 20
     machine: TF= 10, rank= 10
    gradient: TF= 15, rank= 15
      neural: TF= 30, rank= 30
    learning: TF= 50, rank= 50

Register ranks (Level 3, sum of active bits):
  doc_a: 4 active positions → reg-rank sum = 4
  doc_b: 4 active positions → reg-rank sum = 4

HLLSet ranks (Level 4):
  doc_a: degree=4, rank=4
  doc_b: degree=4, rank=4

Compound ranks (Level 5):
  A ∪ B: max(4, 4) = 4
  A ∩ B: min(4, 4) = 4
  Union popcount: 6
  Intersection popcount: 2

Noether steering (A → B):
  |N-D| divergence: 0
  Structural equilibrium: true


## Summary

All five levels, derivatives, Fisher matrix, and observable mask are pure
integer (u64) operations. Every computation maps directly to FPGA gates:
AND, OR, POPCOUNT, ADD, SUB, CMP. No floating-point unit needed.

The four legs of the rank algebra chair:

1. **Static hierarchy** — F, G, H, K, L, M: how rank is built level by level
2. **Dynamic derivatives** — ΔR, Δ²R, Noether steering, Fisher coupling
3. **FPGA-native** — every operation is integer, every value is u64
4. **Depletion and re-masking** — O(θ) mask, sub-lattice degree, feedback loop